# rpart：递归分割与回归树（系统教程）

本 notebook 系统介绍 R 语言 `rpart` 包。整体结构遵循「**包介绍 → 机制 → 方法 → 专题 → 附录**」的主线：

- **第 0 节 预备**：rpart 包是什么、能做什么、**函数全览**、设计特点、安装与基本用法；术语表、控制参数总表、统一环境设置。
- **第 1 节 建树与剪枝机制**：分支原理（不纯度）、cp、xerror、xstd、1-SE 规则、`prune_se`、cp 表逐列解读——所有 method 共用的核心，集中讲透，后续不再重复。
- **第 2 节 按 method 展开**：`class` → `anova` → `poisson` → 生存分析，每个 method 只讲它特有的部分。
- **第 3 节 横切专题**：代价敏感三法、缺失值与 surrogate、变量重要性、时间复杂度——与 method 正交，单独成节。
- **附录**：predict 输出速查、常见坑、**1-SE 规则的深入探讨**、**rpart 其余函数与数据集速览（补全包内全部对象）**。

参考文献：
1. Random Forests with R，Robin Genuer，Jean-Michel Poggi，2020
2. rpart 官方文档 `?rpart`、`?rpart.control`、`?predict.rpart`

算法参见（综合参考，单篇不一定详尽正确）：
- https://zhuanlan.zhihu.com/p/85731206
- https://zhuanlan.zhihu.com/p/139519852
- https://zhuanlan.zhihu.com/p/82054400

# 第 0 节 预备

## 0.1 rpart 包是什么

`rpart`（**R**ecursive **PART**itioning）是 R 中实现 **CART（Classification and Regression Trees）** 算法的主力包，由 Terry Therneau 等人维护，是 R 基础发行版推荐包之一（`recommended`，随 R 一起安装）。

**它解决什么问题：** 给定一张「自变量 + 因变量」的数据表，用一系列**轴平行**的二元分割（如 `x1 < 3.2`、`Type ∈ {F, I}`）把样本递归地切成若干组，使每组内部因变量尽量同质；最终得到一棵树，可用于**预测**和**解释**。因变量是类别 → 分类树；连续 → 回归树；计数 → 泊松树；生存时间 → 生存树。

## 0.2 rpart 包的函数全览（函数地图）

`rpart` 包本身的函数并不多，但五脏俱全，覆盖了从建树到预测的全流程。为了方便查阅，将其核心函数按功能分类如下：

| 功能分类 | 核心函数 | 说明 |
|---|---|---|
| **建树** | `rpart()` | 最核心的函数，负责训练决策树。通过 `method` 参数自动适配分类、回归、生存分析等。 |
| **控制参数** | `rpart.control()` | 用于设定树的生长规则（如 `cp`、`maxdepth`、`minsplit`、`xval` 等），通常在 `rpart()` 内通过 `control = rpart.control(...)` 调用。 |
| **预测** | `predict.rpart()` | 对训练好的树模型进行预测。支持输出类别、概率、连续值等（依据 `type` 参数）。 |
| **剪枝** | `prune()` / `prune.rpart()` | 根据 `cp` 值对一棵充分生长的树进行剪枝，提取子树。（注：本教程为了方便，大量使用了 `treemisc::prune_se()` 自动执行 1-SE 剪枝）。 |
| **查看与摘要** | `print.rpart()`<br>`summary.rpart()` | `print` 快速输出树的分支结构；`summary` 输出详细的模型摘要，包括 cp 表、变量重要性、各节点的分裂优度等。 |
| **可视化** | `plot.rpart()`<br>`text.rpart()`<br>`labels.rpart()`<br>`post()` / `post.rpart()` | `plot` 绘制基础树结构图，`text` 在节点旁添加文字，`labels` 生成分割标签，`post` 输出 PostScript 树图。*(注：业界更推荐 `rpart.plot::rpart.plot()` 或 `partykit::plot(as.party())`)*。 |
| **评估与诊断** | `printcp()`<br>`plotcp()`<br>`rsq.rpart()`<br>`meanvar()` / `meanvar.rpart()`<br>`residuals.rpart()`<br>`xpred.rpart()` | `printcp` 打印 cp 表；`plotcp` 画 xerror 随 cp 曲线（横轴为相邻 cp 几何平均）；`rsq.rpart` 画各分裂的近似 R²；`meanvar` 画节点均值-方差图；`residuals` 提取残差；`xpred` 返回交叉验证预测值。 |
| **交互/辅助** | `path.rpart()`<br>`snip.rpart()`<br>`na.rpart()`<br>`rpart.exp()`<br>`rpart.object` | `path` 提取从根节点到特定节点的路径规则；`snip` 交互式手动剪枝；`na.rpart` 处理缺失值；`rpart.exp` 为指数（生存）拟合初始化；`rpart.object` 是树对象的结构定义。 |
| **数据集** | `kyphosis`<br>`solder` / `solder.balance`<br>`stagec`<br>`car.test.frame`<br>`car90`<br>`cu.summary` | rpart 包自带的经典示例数据集（汽车数据一组三个：`car.test.frame`、`car90`、`cu.summary`）。 |

> **提示**：`rpart` 的很多核心功能（如 cp 表、剪枝逻辑）在其他高级树模型包（如 `randomForest`、`xgboost`）中都有类似的影子，掌握 `rpart` 的函数地图，是入门机器学习树模型的最佳起点。本教程正文重点讲主干函数，**其余函数与数据集的速览见 [附录 D](#appendix-d)**。

## 0.3 设计特点与基本用法

**设计特点（相对其他树包）：**

1. **分类与回归统一接口**：同一函数 `rpart()`，按因变量类型自动选 `method`；自变量可连续可离散，且不要求相互独立。
2. **内置剪枝机制**：通过复杂度参数 `cp` 与交叉验证误差 `xerror` 控制树的大小，避免过拟合（`rpart` 的核心特色）。
3. **代价敏感分类**：支持先验法、代价矩阵法、样本权重法三种手段（见 3.1）。
4. **缺失值处理**：建模时用 surrogate splits 搜索替代分割，预测时按代理逐级启用（见 3.2）。
5. **变量重要性**：自带 `variable.importance`，也可配合 permutation 法（见 3.3）。
6. **输出规范**：`print`/`summary`/`plot`/`predict` 方法齐全，cp 表是理解模型复杂度的核心输出。

**`method` 的自动选择规则（重要）：** `rpart()` 会依据因变量类型自动推断 `method`：

| 因变量类型 | 自动选用的 `method` |
|---|---|
| 因子（factor） | `class`（分类树） |
| 连续数值 | `anova`（回归树） |
| `Surv()` 对象 | `exp`（生存树） |

> **陷阱**：**计数因变量（如 0,1,2,… 的次数）会被当作连续数值而自动走 `anova`**，并不会自动选 `poisson`。若确实要用泊松树，必须**显式指定** `method = "poisson"`（见 2.3）。同理，二分类若因变量是数值 0/1，也会走 `anova`，需先转为因子。

**安装与加载：**

```r
install.packages("rpart")      # 通常已随 R 安装
library(rpart)
```

**最小用法示例：**

```r
fit <- rpart(y ~ ., data = df)              # 建树（默认 cp = 0.01）
print(fit)                                  # 打印树结构
print(fit$cptable)                          # cp 表（核心！见 1.3）
plot(fit); text(fit)                        # 基础绘图
predict(fit, newdata = df, type = "class")  # 预测
```

> 本 notebook 用到的辅助包：`partykit`（`as.party()` 画更清晰的树）、`treemisc`（`prune_se()`、`gen_friedman1()`、`mushroom` 等）、`rpart.plot`/`rattle`（`fancyRpartPlot`）、`skimr`（数据勘察）、`AppliedPredictiveModeling`（abalone 数据）。

## 0.4 术语表

| 术语 | 英文 | 含义 |
|---|---|---|
| 复杂度参数 | cp (complexity parameter) | 控制树生长/剪枝的阈值。cp 越小树越大；cp=0 表示充分生长；cp=Inf 表示只有根节点 |
| 相对误差 | rel error | 训练集误差（以根节点误差为基准的相对值） |
| 交叉验证误差 | xerror | k 折交叉验证误差的均值（默认 10 折） |
| 交叉验证标准差 | xstd | xerror 的标准差，反映其随机性 |
| 1-SE 规则 | 1-SE rule | 在满足 `xerror ≤ min(xerror) + xstd` 的 cp 中取最大者 |
| 分割优度 | improve | 一次分割带来的 impurity（或误差）下降量 |
| 代理分割 | surrogate split | 当主分割变量缺失时，用与主分割一致性最高的其他变量代替 |
| 一致性 | agree / adj | 代理分割与原分割的一致比例 / 调整后一致比例 |
| 不纯度 | impurity | 分类中用 Gini 指数；回归中用方差（SSE） |

## 0.5 `rpart.control` 控制参数总表

| 参数 | 默认值 | 作用 |
|---|---|---|
| `cp` | 0.01 | 复杂度参数；分割必须使误差下降至少 cp 倍才保留 |
| `minsplit` | 20 | 节点至少含多少个样本才尝试分割 |
| `minbucket` | `minsplit/3` | 叶节点至少含多少个样本 |
| `maxdepth` | 30 | 树的最大深度 |
| `maxcompete` | 4 | 每个节点保留多少个竞争分割（用于输出） |
| `maxsurrogate` | 5 | 每个节点最多搜索多少个代理分割（0 表示不搜索，可加速） |
| `usesurrogate` | 2 | 0=只用主分割，缺失则不分；1=按序用代理，全缺失则不分；2=全缺失则按多数类返回 |
| `xval` | 10 | 交叉验证折数（用于计算 xerror/xstd） |
| `surrogatestyle` | 0 | 代理分割的选择风格（0=按 agree 排序，1=按 adj 排序） |

> 提示 1：`xerror` 与 `xstd` 具有随机性（源于数据随机分割）。复现时请在 `rpart()` 前 `set.seed()`。
>
> 提示 2：分类树还可通过 `rpart()` 的 `parms` 参数控制不纯度与先验：`parms = list(split = "gini")` 或 `split = "information"` 选择 Gini 指数还是信息增益作为分裂准则；`parms = list(prior = ...)`、`parms = list(loss = ...)` 用于代价敏感（见 3.1）。`parms` 不属于 `rpart.control`，而是 `rpart()` 的直接参数。

## 0.6 关于 `prune_se()`

本 notebook 多次使用 `prune_se()`（来自 `treemisc` 包）。它按 **1-SE 规则** 从一棵充分生长的树中剪出最优子树：

- `prune = TRUE`：返回剪枝后的树对象；
- `se = 1`：使用 1 倍标准差（即标准 1-SE 规则）；`se = 0` 则退化为「取 xerror 最小的 cp」。

等价于手工：在 cp 表中找 `xerror ≤ min(xerror) + se*xstd` 的最大 cp，再 `prune(tree, cp = 该值)`。

## 0.7 统一环境设置

Jupyter 的 kernel 是持续的，环境设置只需在开头做一次，后续代码块不再重复 `p_load` 和 `options`（新包首次出现时再单独 `p_load`）。

**注意：** R 语言基础绘图默认就是“白底黑字”。为了确保导出图片的兼容性，这里恢复了所有图表的默认白色背景。如果你在 Jupyter 暗色主题下看，图片本身依然是白底的。

In [ ]:
library(pacman)                                      # 加载包管理工具

options(warn = -1)                                   # 忽略一切警告（教学演示用）
options("width" = 140)                               # 充分利用打印宽度
options(repr.plot.width = 16, repr.plot.height = 9)  # 设置画幅比例 16:9

p_load(rpart, partykit, treemisc, skimr, rpart.plot, rattle) # 批量加载所需包
s <- skim_tee                                        # 勘察数据的简写

# 第 1 节 建树与剪枝机制

本节用二维平面上的分类/回归直观例子引入，但**只讲机制**：树如何分段拟合、分支的依据是什么、cp 如何控制复杂度、cp 表怎么读、1-SE 规则怎么用。这些是所有 method 共用的，第 2 节各 method 不再重复。

## 1.1 直观引入：树能分段拟合非线性边界

构造一个两类分类问题：真实分类界限为曲线 $x_2 = x_1\cos(x_1)$，两类分别在曲线上下各偏移 4 并加噪声。决策树用**轴平行**的分割逼近这条曲线。

In [ ]:
n <- 10000                                           # 设定样本量
u <- 10                                              # 自变量取值范围 [-10, 10]
v <- 5                                               # 噪声强度

# -------------------- 构建训练集 cl_tr --------------------
set.seed(11)                                         # 固定随机种子，保证结果可复现
x1 <- sort(runif(n, min = -u, max = u))              # 生成 n 个均匀分布的 x1，并从小到大排序
x2 <- x1 * cos(x1) + v * rnorm(n) - 4                # 根据理论曲线生成 x2，并向下偏移 4，加入正态噪声
d0 <- data.frame(y = 1, x1, x2)                      # 组合成第一类数据 (y=1)

set.seed(22)                                         # 固定随机种子
x1 <- sort(runif(n, min = -u, max = u))              # 重新生成 x1
x2 <- x1 * cos(x1) + v * rnorm(n) + 4                # 根据理论曲线生成 x2，并向上偏移 4，加入正态噪声
d1 <- data.frame(y = 2, x1, x2)                      # 组合成第二类数据 (y=2)

cl_tr <- rbind(d0, d1)                               # 合并两类数据，共 2n 行
cl_tr$y <- factor(cl_tr$y)                           # 将目标变量转为因子类型，以便进行分类树建模

# -------------------- 绘制训练集散点图 --------------------
plot(cl_tr$x1, cl_tr$x2, col = cl_tr$y, xlab = "x1", ylab = "x2", cex = 0.5) # 画散点图，cex 减小点的大小防重叠
curve(x * cos(x), col = "red", lwd = 6, lty = 3, add = TRUE)                  # 画出真实的分类界限
legend("topright", "理论分类界限", lwd = 6, col = "red", lty = 3, cex = 0.8)   # 放在右上角，避免遮挡
grid()                                               # 添加网格线

# -------------------- 构建测试集 cl_te --------------------
set.seed(33)                                         # 固定随机种子
x1 <- sort(runif(n, min = -u, max = u))              # 生成 x1
x2 <- x1 * cos(x1) + v * rnorm(n) - 4                # 生成 x2（第一类）
d0 <- data.frame(y = 1, x1, x2)                      # 组合第一类数据

set.seed(44)                                         # 固定随机种子
x1 <- sort(runif(n, min = -u, max = u))              # 生成 x1
x2 <- x1 * cos(x1) + v * rnorm(n) + 4                # 生成 x2（第二类）
d1 <- data.frame(y = 2, x1, x2)                      # 组合第二类数据

cl_te <- rbind(d0, d1)                               # 合并测试集
cl_te$y <- factor(cl_te$y)                           # 目标变量转因子

## 1.2 建树的核心原理：分支的依据是什么？

决策树在每一个节点上，都会面临一个核心问题：**“我应该在哪个变量的哪个值上切一刀，才能让切出来的两拨人最纯粹？”**

为了衡量“纯粹程度”，CART 算法引入了**不纯度（Impurity）**的概念：

- **分类树**：使用 **Gini 指数**。$G = 1 - \sum p_k^2$。如果节点里只有一类，$G=0$（最纯）；如果两类各占 50%，$G=0.5$（最杂）。
- **回归树**：使用 **方差（MSE）**。如果节点里所有 $y$ 值都一样，方差为 0（最纯）。

### 寻找最优切分点（启发式搜索）
1. 遍历当前节点所有可能的自变量（如 $x_1, x_2$）。
2. 对每个自变量，把它的所有可能取值作为候选切点（如 $x_1=0.5, x_1=1.2$...），尝试切一刀。
3. 分别计算切分后左右子节点的加权不纯度：$\text{Imp}_{\text{new}} = \frac{n_L}{n} G_L + \frac{n_R}{n} G_R$。
4. 计算**不纯度下降量（Improve）** = `父节点不纯度 - 切分后子节点加权不纯度`。
5. **选择使 Improve 最大的那个变量的那个切点，作为最终的分支。**

这就是 `rpart` 生成树结构的根本依据。

## 1.3 cp 表逐列解读（核心技能）

用 `cp = 0` 让树充分生长，得到完整的 cp 表。**这是 rpart 最核心的读表技能**，后面各 method 的 cp 表读法完全一致。

为了看清所有被省略的中间行，这里使用 `as.data.frame()` 将 cp 表转换为数据框输出。

In [ ]:
set.seed(2)                                          # 固定 xstd：因为交叉验证数据是随机分割的
ct <- rpart(y ~ ., data = cl_tr, method = "class", cp = 0) # cp=0 让树充分生长（不断分裂直到长不动为止）
as.data.frame(ct$cptable)                            # 以数据框格式在 Jupyter 中展示完整的 cp 表

以上输出的结果分为两类，计算逻辑完全不同：

**第一类：基于全量训练集直接算出来的（无需交叉验证）**
* **`CP`**：算法设定的复杂度阈值。
* **`nsplit`**：这棵树实际发生的分裂次数。
* **`rel error`**：相对训练误差。直接用全量训练集预测全量训练集算出来的误差。
  * 注意看第 1 行（根节点），`rel error` 是 1.0000。因为树还没分叉，所有样本都被预测为多数类，误差最大。往下走树越来越复杂，`rel error` 降到 0.3121，它在训练集上越来越准。

**第二类：通过 10 折交叉验证得到的**
* **`xerror`**：10 折交叉验证误差的**均值**。为了防止模型“死记硬背”训练集，R 把数据切成 10 份，轮流用 9 份训练、1 份验证，算 10 次误差并求平均。
* **`xstd`**：10 折交叉验证误差的**标准差**。因为切了 10 次，有波动，用标准差衡量波动的范围。
  * 可以看到，`xerror` 是一条**先降后升的 U 型曲线**。在树较小时（上面几行），它变小；当树极度复杂时（最后几行），它反弹到 0.50 左右，这正是**过拟合**的信号。

**逐列含义与读法：**

| 列 | 含义 | 怎么读 |
|---|---|---|
| `CP` | 复杂度参数 | 从大到小排列；每行对应一棵子树 |
| `nsplit` | 该树的分割数（内部节点数） | 随 CP 下降而上升，树越来越大 |
| `rel error` | 训练集相对误差 = 训练误差 / 根节点误差 | 随 CP 下降单调下降（树越大训练越准） |
| `xerror` | 交叉验证误差均值 | 先降后升——升即过拟合信号 |
| `xstd` | xerror 的标准差 | 反映 xerror 的波动；1-SE 规则用它 |

**1-SE 规则：** 令 `xerror_min = min(xerror)`，阈值 `thr = xerror_min + 1*xstd[对应行]`，在满足 `xerror ≤ thr` 的所有 cp 中取**最大**的 cp。

> 为什么这样设定阈值？其背后的统计学原理和业务比喻请参阅 **[附录 C：关于 1-SE 规则的深入探讨](#appendix-c)**。

In [ ]:
plotcp(ct)                                           # 横轴是相邻 cp 的几何平均值（取值可介于二者之间）
grid()                                               # 添加网格线
# 水平虚线 = min(xerror) + 1*xstd；与之相交最左边的点即 1-SE 最优 cp

## 1.4 三棵树对比：完全树 / 最简树 / 1-SE 最优树

用同一个例子展示 cp 对树复杂度与泛化性能的控制。

In [ ]:
# 辅助函数：画分类界限（0.5 等高线）
draw_boundary <- function(tree, cl, col = "blue") {
  x1g <- seq(min(cl$x1), max(cl$x1), length = 100)   # 生成网格 x1 坐标
  x2g <- seq(min(cl$x2), max(cl$x2), length = 100)   # 生成网格 x2 坐标
  pd <- function(a, b) predict(tree, data.frame(x1 = a, x2 = b), type = "prob")[, 1] # 预测属于第一类的概率
  z <- outer(x1g, x2g, FUN = pd)                     # 在网格上计算概率矩阵
  contour(x1g, x2g, z, levels = 0.5, drawlabels = FALSE, lwd = 6, col = col, add = TRUE) # 画概率为 0.5 的等高线
}

**完全树**（`cp = 0`）：充分生长，分类界限复杂。

In [ ]:
ct1 <- rpart(y ~ ., data = cl_tr, method = "class", cp = 0) # 完全树
plot(cl_tr$x1, cl_tr$x2, col = cl_tr$y, xlab = "x1", ylab = "x2", cex = 0.5) # 画散点图
draw_boundary(ct1, cl_tr)                            # 画出完全树的分类界限
curve(x * cos(x), col = "red", lwd = 6, lty = 3, add = TRUE)                  # 画理论界限
legend("topright", c("理论界限", "cp=0 完全树"), lwd = 6, col = c("red", "blue"), lty = c(3, 1), cex = 0.8) # 添加图例
grid()                                               # 添加网格

**最简树**（`cp = Inf`）：只有根节点，所有样本判为同一类，无分类界限。

In [ ]:
ct2 <- rpart(y ~ ., data = cl_tr, method = "class", cp = Inf) # 最简树
plot(cl_tr$x1, cl_tr$x2, col = cl_tr$y, xlab = "x1", ylab = "x2", cex = 0.5) # 画散点图
curve(x * cos(x), col = "red", lwd = 6, lty = 3, add = TRUE)                  # 画理论界限
legend("topright", c("理论界限", "cp=Inf 仅根节点"), lwd = 6, col = c("red", "blue"), lty = c(3, 1), cex = 0.8) # 添加图例
grid()                                               # 添加网格
plot(as.party(ct2))                                  # 使用 partykit 画出树的结构

**1-SE 最优树**：用 `prune_se()` 剪枝，分类界限接近理论曲线。

In [ ]:
ct3 <- prune_se(ct1, prune = TRUE, se = 1)           # 根据 1-SE 规则剪枝得到最优树
plot(cl_tr$x1, cl_tr$x2, col = cl_tr$y, xlab = "x1", ylab = "x2", cex = 0.5) # 画散点图
draw_boundary(ct3, cl_tr)                            # 画出最优树的分类界限
curve(x * cos(x), col = "red", lwd = 6, lty = 3, add = TRUE)                  # 画理论界限
legend("topright", c("理论界限", "1-SE 最优树"), lwd = 6, col = c("red", "blue"), lty = c(3, 1), cex = 0.8) # 添加图例
grid()                                               # 添加网格
plot(as.party(ct3))                                  # 画出树的结构

**三棵树的泛化性能对比：**

In [ ]:
err <- function(tree) c(
  train = mean(predict(tree, cl_tr, type = "class") != cl_tr$y), # 计算训练集错误率
  test  = mean(predict(tree, cl_te, type = "class") != cl_te$y)  # 计算测试集错误率
)
rbind(cp0 = err(ct1), cpInf = err(ct2), best1SE = err(ct3))       # 汇总三棵树的错误率

可见 1-SE 最优树的测试集误差最低——**cp 控制模型复杂度，1-SE 规则在「训练更准」与「泛化更好」之间取平衡**。

> 注意：本节的完全树用了 `cp = 0`。在普通分类/回归场景，`cp = 0` 是常见的「充分生长」写法；但在**代价敏感**场景（3.1），`cp = 0` 可能因权重/先验的浮点边界导致结果不稳定，那时建议用 `cp = 1e-5`。

> **本节要点**
> 1. **建树依据**：通过遍历自变量和切点，找到能使“不纯度下降量（Improve）”最大的切分方式。
> 2. cp 表五列：CP、nsplit、rel error（训练）、xerror（CV）、xstd（CV 标准差）。
> 3. xerror 先降后升；上升即过拟合。
> 4. 1-SE 规则：`xerror ≤ min(xerror) + xstd` 中取最大 cp；`prune_se(tree, se = 1)` 即此操作。
> 5. 完全树训练误差低但测试误差高；最简树欠拟合；1-SE 树泛化最优。

# 第 2 节 按 method 展开

`rpart` 根据因变量类型自动选择 `method`：因子 → `class`；连续数值 → `anova`；计数 → `poisson`；`Surv()` 对象 → `exp`（生存）。本节逐个展开，每个 method 只讲它**特有**的部分，cp 表读法、1-SE 规则直接沿用第 1 节。

> **再次强调**：计数因变量默认会被当作连续而走 `anova`，要得到泊松树必须显式 `method = "poisson"`（见 2.3）。

## 2.1 `method = "class"`：分类树

### 2.1.1 平衡、代价不敏感、混合自变量的二分类

abalone 数据集：4177 只鲍鱼，自变量含离散（Type：公/母/幼）与连续（尺寸、重量），因变量 Rings（环数）。将 Rings 按 >9 分为 L/H 两类（数量大致平衡）。

In [ ]:
p_load(AppliedPredictiveModeling)                    # 加载含有 abalone 数据集的包
data(abalone)                                        # 导入数据
ab <- abalone                                        # 备份数据
ab$Rings <- factor(ab$Rings > 9, labels = c("L", "H")) # 将环数二值化：>9 为 H，否则为 L
s(ab)                                                # 勘察数据

In [ ]:
set.seed(100)                                        # 固定随机种子，保证切分一致
i <- sample(nrow(ab), nrow(ab) * 0.5)                # 随机抽取一半作为训练集索引
tr_ab <- ab[i, ]; te_ab <- ab[-i, ]                  # 划分训练集和测试集

set.seed(2)                                          # 固定交叉验证随机种子
ct_ab <- rpart(Rings ~ ., data = tr_ab, cp = 0)      # 建立分类树，cp=0 充分生长
print(ct_ab$cptable)                                 # 打印 cp 表（或改用 as.data.frame）

cp 表读法同 1.3。`summary()` 还会输出各节点的分割详情（Primary splits、Surrogate splits、improve 值）。

In [ ]:
summary(ct_ab)                                       # 查看模型详细摘要

In [ ]:
plotcp(ct_ab); grid()                                # 画出 cp 曲线
ct_ab_best <- prune_se(ct_ab, prune = TRUE, se = 1)  # 1-SE 剪枝得最优树
plot(as.party(ct_ab_best))                           # 画出最优树的结构图

In [ ]:
pred_tr <- predict(ct_ab_best, tr_ab, type = "class") # 预测训练集
pred_te <- predict(ct_ab_best, te_ab, type = "class") # 预测测试集
c(train_error = mean(pred_tr != tr_ab$Rings),          # 训练集错误率
  test_error  = mean(pred_te != te_ab$Rings))          # 测试集错误率

### 2.1.2 `class` 的 impurity 与 improve

分类树用 **Gini 指数**作不纯度。节点不纯度 $G = 1 - \sum_k p_k^2$，其中 $p_k$ 为节点内第 $k$ 类比例。

一次分割的 improve（分割优度）= `n·G_parent − (n_L·G_left + n_R·G_right)`。

以根节点按 `ShellWeight < 0.2545` 分割为例手算验证：

In [ ]:
tab <- with(tr_ab, table(cut(ShellWeight, c(0, 0.2545, Inf)), Rings)) # 按 0.2545 离散化后统计交叉表
print(addmargins(tab))                               # 打印带边际和的表格

gini <- function(n1, n2) 1 - (n1/(n1+n2))^2 - (n2/(n1+n2))^2 # 定义 Gini 指数计算函数
n  <- sum(tab); nL <- sum(tab[1, ]); nR <- sum(tab[2, ])      # 计算父节点、左子节点、右子节点的样本数
imp <- n*gini(tab[1,1]+tab[2,1], tab[1,2]+tab[2,2]) -         # 父节点不纯度乘样本数
       nL*gini(tab[1,1], tab[1,2]) - nR*gini(tab[2,1], tab[2,2]) # 减去左右子节点不纯度乘样本数
cat("手算 improve =", round(imp, 4), "\n")            # 输出手算结果

与 `summary(ct_ab)` 中根节点 `improve` 一致（数值舍入差异源于软件内部精度）。

> 说明：分类树的分裂准则默认为 Gini 指数；也可通过 `parms = list(split = "information")` 改用信息增益，此时 improve 的计算口径相应变为熵的下降量。

### 2.1.3 `class` 的 predict 输出

- `type = "class"`：预测类别（因子）；
- `type = "prob"`：各类别概率矩阵（列顺序 = `levels(y)`）。

```r
predict(ct_ab_best, te_ab, type = "class")  # 因子
predict(ct_ab_best, te_ab, type = "prob")   # 矩阵，列 = levels
```

### 2.1.4 多分类：如何读决策树图

因变量有二个以上类别时，叶节点显示各类比例，节点标签显示多数类。以 iris 为例：

In [ ]:
s(iris)                                              # 勘察 iris 数据
iris_best <- prune_se(rpart(Species ~ ., data = iris), prune = TRUE, se = 1) # 1-SE 剪枝
plot(as.party(iris_best))                            # 画多分类树图

**读图要点**：`as.party()` 画出的每个叶节点方框里会显示三类（setosa / versicolor / virginica）的样本比例与预测类别。例如某个叶节点显示 `(0, 0.9, 0.1)`，表示该节点内 90% 为 versicolor、10% 为 virginica，节点标签为 versicolor。

**多分类的 predict 输出**（概率矩阵列为三个类别，可据此看模型在边界样本上的不确定程度）：

In [ ]:
head(predict(iris_best, iris, type = "prob"), 6)     # 查看前 6 个样本的三类概率
table(predict(iris_best, iris, type = "class"), iris$Species) # 混淆矩阵对照

> **本节要点**
> 1. `class` 用 Gini 不纯度；improve = n·G_parent − Σ n_child·G_child；可用 `parms = list(split="information")` 改用信息增益。
> 2. `predict` 的 `type = "class"/"prob"` 分别返回类别/概率矩阵。
> 3. 多分类叶节点显示各类比例，节点标签为多数类；`type="prob"` 可查看各类概率。

## 2.2 `method = "anova"`：回归树

因变量连续时自动用 `anova`。不纯度用**方差（SSE）**，improve = `SST − (SSL + SSR)`。

### 2.2.1 直观：回归曲线

真实曲线 $y = x\sin(x)$ 加噪声，树用分段常数逼近。

In [ ]:
n <- 1000                                            # 样本量
set.seed(100)                                        # 固定随机种子
x <- sort(runif(n, -2*pi, pi)); y <- x*sin(x) + 1.2*rnorm(n) # 生成带噪声的回归训练集
re_tr <- data.frame(x, y)                            # 组合训练集

set.seed(200)                                        # 固定随机种子
x <- runif(n, -2*pi, pi); y <- x*sin(x) + 1.2*rnorm(n)       # 生成带噪声的回归测试集
re_te <- data.frame(x, y)                            # 组合测试集

plot(re_tr$x, re_tr$y, cex = 0.5, xlab = "x", ylab = "y")   # 画出训练集散点
curve(x*sin(x), -2*pi, pi, lwd = 6, col = "red", lty = 3, add = TRUE) # 画出真实曲线
legend("topright", "理论曲线", lwd = 6, col = "red", lty = 3, cex = 0.8) # 添加图例
grid()                                               # 添加网格

In [ ]:
set.seed(2)                                          # 固定交叉验证随机种子
rt <- rpart(y ~ ., data = re_tr, cp = 0)             # y 连续，自动采用 anova 方法建树
plotcp(rt); grid()                                   # 画出 cp 曲线
rt_best <- prune_se(rt, prune = TRUE, se = 1)        # 剪枝得到最优回归树

plot(re_tr$x, re_tr$y, cex = 0.5, xlab = "x", ylab = "y")   # 画出训练集散点
lines(re_tr$x, predict(rt_best, re_tr), lwd = 4, col = "blue") # 画出回归树的预测折线
curve(x*sin(x), -2*pi, pi, lwd = 6, col = "red", lty = 3, add = TRUE) # 画出真实曲线
legend("topright", c("理论", "1-SE 回归树"), lwd = c(4, 6), col = c("red", "blue"), lty = c(3, 1)) # 添加图例
grid()                                               # 添加网格

### 2.2.2 `anova` 的 predict 输出

- `type = "vector"`（默认）：预测均值向量；
- `type = "matrix"`：`nsplit × n` 矩阵。

```r
predict(rt_best, re_te)            # 向量
predict(rt_best, re_te, type="matrix")  # 矩阵
```

### 2.2.3 自变量混合取值：abalone 回归

同一个 abalone 数据，因变量改为连续的 Rings（不二值化），自变量仍混合离散/连续。

In [ ]:
ab2 <- abalone                                       # Rings 保持数值
set.seed(100)                                        # 固定随机种子
I <- sample(nrow(ab2), nrow(ab2) * 0.5)              # 随机抽取一半作为训练集索引
tr2 <- ab2[I, ]; te2 <- ab2[-I, ]                    # 划分训练集和测试集

set.seed(2)                                          # 固定交叉验证随机种子
rt_ab <- rpart(Rings ~ ., data = tr2, cp = 0)        # 建立回归树
plotcp(rt_ab); grid()                                # 画出 cp 曲线
rt_ab_best <- prune_se(rt_ab, prune = TRUE, se = 1)  # 1-SE 剪枝
plot(as.party(rt_ab_best))                           # 画出最优回归树

In [ ]:
rmse <- function(pred, obs) sqrt(mean((pred - obs)^2)) # 计算均方根误差 (RMSE)
c(train_RMSE = rmse(predict(rt_ab_best, tr2), tr2$Rings), # 训练集 RMSE
  test_RMSE  = rmse(predict(rt_ab_best, te2), te2$Rings)) # 测试集 RMSE

### 2.2.4 cp 对回归性能的影响

遍历 cp 表各 cp 值，画训练/测试 RMSE 曲线，验证 1-SE 最优 cp 的合理性。

In [ ]:
cp <- rt_ab$cptable                                  # 提取 cp 表
a <- cp[, 1]; L <- length(a)                         # 提取 cp 表第一列及长度
Cp <- sqrt(a[-L] * a[-1])                            # 计算相邻 cp 的几何平均
node <- cp[-1, 2] + 1                                # 计算对应的叶节点数 (nsplit + 1)

err_tr <- err_te <- numeric(length(Cp))              # 初始化用于存储误差的向量
for (k in seq_along(Cp)) {                           # 遍历每一个 cp 值
  m <- rpart(Rings ~ ., data = tr2, cp = Cp[k])      # 用当前 cp 建树
  err_tr[k] <- rmse(predict(m, tr2), tr2$Rings)      # 计算训练集 RMSE
  err_te[k] <- rmse(predict(m, te2), te2$Rings)      # 计算测试集 RMSE
}

p_load(plotly)                                       # 加载 plotly 绘制交互式图形
plot_ly(x = Cp) %>%
  add_trace(y = err_tr, type = "scatter", name = "train", mode = "lines+markers", color = I("black")) %>% # 添加训练集折线
  add_trace(y = err_te, type = "scatter", name = "test",  mode = "lines+markers", color = I("red")) %>%   # 添加测试集折线
  layout(title = "RMSE vs cp", xaxis = list(title = "cp"), yaxis = list(title = "RMSE"))               # 设置标题和轴标签

> **本节要点**
> 1. `anova` 用 SSE/方差作不纯度；improve = SST − (SSL+SSR)。
> 2. `predict` 默认 `type = "vector"` 返回均值；`"matrix"` 返回节点级矩阵。
> 3. cp 越小训练 RMSE 越低、测试 RMSE 先降后升；1-SE cp 位于测试曲线拐点附近。

## 2.3 `method = "poisson"`：计数因变量

因变量为计数时用 `poisson`。improve = 似然比统计量：`D_parent − (D_left + D_right)`，D 为 Poisson 偏差。

> **必须显式指定**：计数因变量若直接写数值，`rpart()` 会自动走 `anova`，不会自动选 `poisson`。因此下面建树时显式写 `method = "poisson"`。

solder.balance 数据集：skips（焊接缺陷数）对 Opening、Solder、Mask、PadType、Panel 建模。

In [ ]:
s(solder.balance)                                    # 勘察数据
set.seed(2)                                          # 固定交叉验证随机种子
sfit <- rpart(skips ~ Opening + Solder + Mask + PadType + Panel,
              data = solder.balance, method = "poisson") # 显式指定泊松树
plotcp(sfit); grid()                                 # 画出 cp 曲线
sfito <- prune_se(sfit, prune = TRUE, se = 1)        # 1-SE 剪枝
plot(as.party(sfito))                                # 画出最优树

**improve 的手算对照**：泊松树用 Poisson 偏差 D 衡量不纯度。对节点内计数 $y_i$、均值 $\bar y$，偏差 $D = 2\sum [y_i\log(y_i/\bar y) - (y_i-\bar y)]$（$y_i=0$ 项取 0）。一次分割的 improve = `D_parent − (D_left + D_right)`。以根节点首刀为例，用 `summary()` 的 improve 与手算对照：

In [ ]:
summary(sfit)                                        # 查看根节点 Primary splits 的 improve

In [ ]:
pois_dev <- function(y) {                            # 节点 Poisson 偏差
  ybar <- mean(y)
  if (ybar == 0) return(0)
  t <- ifelse(y == 0, 0, y * log(y / ybar))
  2 * sum(t - (y - ybar))
}
first_var <- "Opening"                              # 以 summary 中根节点主分割变量为例
cutv <- summary(sfit)$splits[1, "index"]             # 取该变量的分割点（可能为类别组合，视数据而定）
# 若主分割为数值切点，可直接按切点分左右；若为类别组合，可按水平拆分
lv <- levels(solder.balance[[first_var]])            # 该变量的水平
idx <- summary(sfit)$splits[1, "index"]              # 分割索引
left_lv <- lv[idx]                                   # 左支水平（按 rpart 的 index 约定）
left  <- solder.balance[[first_var]] %in% left_lv    # 左支样本
y <- solder.balance$skips
imp <- pois_dev(y) - pois_dev(y[left]) - pois_dev(y[!left]) # 手算 improve
cat("手算 improve =", round(imp, 4), "\n")            # 与 summary 对照（舍入差异属正常）

> 提示：solder.balance 的自变量均为类别型，rpart 对类别变量会在水平组合中搜索最优二分；`summary()$splits` 的 `index` 编码了左支水平，具体拆分方式以 `summary()` 输出的方向说明为准。手算结果与 `summary()` 的 improve 在同一量级即可验证口径正确。

**`poisson` 的 `predict` 输出**：

In [ ]:
phat <- predict(sfito, solder.balance)               # 默认 type="vector"，返回预测计数均值
head(phat)                                           # 查看前几个预测值
pm <- predict(sfito, solder.balance, type = "matrix") # type="matrix" 返回 nsplit × n 矩阵
dim(pm)                                              # 矩阵维度

> **本节要点**
> 1. `poisson` 用 Poisson 偏差的似然比统计量作 improve：`D_parent − (D_left + D_right)`；计数因变量必须显式指定 `method = "poisson"`。
> 2. `predict` 默认 `type = "vector"` 返回预测计数均值；`"matrix"` 返回节点级矩阵。
> 3. cp 表读法、1-SE 规则与 class/anova 完全一致。

## 2.4 生存分析：`Surv()` 因变量

因变量为 `survival::Surv(time, status)` 时，rpart 自动用 `method = "exp"`（指数生存模型）。`Surv` 参数：`time` 随访时间，`event` 状态（0=存活/删失，1=死亡）。

生存树的 improve 基于**指数模型的偏差（deviance）下降**：一次分割的 improve = `D_parent − (D_left + D_right)`，D 由各节点的生存分布（指数似然）计算，口径与 poisson 的偏差下降类似，只是分布换成了生存模型。

In [ ]:
s(stagec)                                            # 勘察前列腺癌数据
set.seed(2)                                          # 固定交叉验证随机种子
pfit <- rpart(survival::Surv(pgtime, pgstat) ~ age + eet + g2 + grade + gleason + ploidy,
              data = stagec)                         # 建立生存树
pfit$method                                          # 确认 method 为 exp

In [ ]:
plotcp(pfit); grid()                                 # 画出 cp 曲线
pfito <- prune_se(pfit, prune = TRUE, se = 1)        # 1-SE 剪枝
plot(as.party(pfito))                                # 画出最优生存树

**生存树的 `predict` 输出**：

- `type = "vector"`（默认）：返回每个样本的预测生存时间（指数模型下的期望生存时间）；
- `type = "matrix"`：返回 `nsplit × n` 节点级矩阵。

In [ ]:
st <- predict(pfito, stagec)                         # 默认返回预测生存时间向量
head(st)                                             # 查看前几个预测生存时间
sm <- predict(pfito, stagec, type = "matrix")        # 节点级矩阵
dim(sm)                                              # 矩阵维度

> **本节要点**
> 1. 因变量写 `Surv(time, event)`，method 自动为 `exp`；improve 基于指数模型偏差下降。
> 2. `predict` 默认 `type = "vector"` 返回预测生存时间；`"matrix"` 返回节点级矩阵。
> 3. 其余流程（cp 表、1-SE、prune_se）不变。

# 第 3 节 横切专题

本节主题与 method 正交：代价敏感、缺失值与 surrogate、变量重要性、时间复杂度。

## 3.1 代价敏感：先验法、代价法、权重法

非平衡数据 + 错判代价不对称时，rpart 提供三种等价手段：

- **先验法**：`parms = list(prior = ...)`，按代价调整类先验；
- **代价法**：`parms = list(loss = ...)`，直接给代价矩阵；
- **权重法**：`weights = ...`，给每个样本 case weight。

**等效条件**：先验法要求 prior 按代价矩阵调整（`prior_k ∝ π_k · Σ_j C_{kj}`）；权重法要求 case weight 与代价成比例（`w_i ∝ C_{y_i, ·}`）。满足时三者数学等价——**直观地说，就是把「错判成本高」的信息，分别折算进「类先验」「损失矩阵」或「样本权重」里**，本质都是让树在分裂和判类时更重视高代价的少数类。

> **先说坑**：本节建树时把 `cp` 设为 `1e-5` 而不是 0。原因是在代价敏感场景下，cp=0 的停止条件对样本权重/先验敏感，边界节点的分裂判定会因浮点与权重不同而不稳定，三种方法得到的树可能不完全一致。第 1 节普通场景用的 `cp=0` 在此不适用。

构造非平衡数据：Rings > 12 为 H（少数类），否则 L（多数类，约 5:1）。设 H→L 代价 5、L→H 代价 1。

In [ ]:
ab3 <- abalone                                       # 备份数据
ab3$Rings <- factor(ab3$Rings > 12, labels = c("L", "H")) # 构造非平衡数据集
s(ab3)                                               # 确认类别比例

set.seed(100)                                        # 固定随机种子
i <- sample(nrow(ab3), nrow(ab3) * 0.5)              # 随机抽取一半作为训练集索引
tr3 <- ab3[i, ]; te3 <- ab3[-i, ]                    # 划分训练集和测试集

cm <- matrix(c(0, 1, 5, 0), nrow = 2, byrow = TRUE)  # 代价矩阵：行=真实类，列=预测类。H判L罚5，L判H罚1
ctrl <- rpart.control(cp = 1e-5)                     # 降低 cp 使树充分生长（规避 cp=0 时的计算不稳定）

### 3.1.1 先验法

In [ ]:
set.seed(22)                                         # 固定交叉验证随机种子
pL <- mean(tr3$Rings == "L")                         # 计算 L 类的实际比例
pr <- c(pL, (1 - pL) * 5); pr <- pr / sum(pr)        # 将 H 类先验乘以代价比 5，并归一化
ct1c <- rpart(Rings ~ ., data = tr3, parms = list(prior = pr), control = ctrl) # 用调整后的先验建树
ct1o <- prune_se(ct1c)                               # 1-SE 剪枝
loss1 <- sum(table(te3$Rings, predict(ct1o, te3, type = "class")) * cm) # 测试集总损失
loss1

### 3.1.2 代价法

In [ ]:
set.seed(22)                                         # 固定交叉验证随机种子
ct2c <- rpart(Rings ~ ., data = tr3, parms = list(loss = cm), control = ctrl) # 直接传入代价矩阵建树
ct2o <- prune_se(ct2c)                               # 1-SE 剪枝
loss2 <- sum(table(te3$Rings, predict(ct2o, te3, type = "class")) * cm) # 测试集总损失
loss2

### 3.1.3 权重法（最简洁）

In [ ]:
w <- ifelse(tr3$Rings == "H", 5, 1)                  # 对少数类 H 赋予 5 倍的样本权重
set.seed(22)                                         # 固定交叉验证随机种子
ct3c <- rpart(Rings ~ ., data = tr3, weights = w, control = ctrl) # 用样本权重建树
ct3o <- prune_se(ct3c)                               # 1-SE 剪枝
loss3 <- sum(table(te3$Rings, predict(ct3o, te3, type = "class")) * cm) # 测试集总损失
loss3

**三法结果并列对比**（总损失越接近说明等效性越好）：

In [ ]:
rbind(prior = loss1, loss = loss2, weights = loss3)  # 三法测试集总损失并列
table(prior = predict(ct1o, te3, type = "class"),
      loss  = predict(ct2o, te3, type = "class"))    # 先验法 vs 代价法的预测一致性
table(prior = predict(ct1o, te3, type = "class"),
      weights = predict(ct3o, te3, type = "class"))  # 先验法 vs 权重法的预测一致性

**关于 cp = 0 的反常**：若把 `ctrl` 的 `cp` 置为 0，三种方法得到的树可能不完全一致。原因是 cp=0 时停止条件对样本权重/先验敏感，边界节点的分裂判定会因浮点与权重而不同。**建议代价敏感场景用 `cp = 1e-5` 而非 0**（第 1 节普通场景的 cp=0 不适用于此）。

> **本节要点**
> 1. 先验法、代价法、权重法在等效条件下结果一致；权重法最直观。
> 2. 代价敏感场景避免 `cp = 0`，用 `cp = 1e-5` 更稳定（第 1 节 cp=0 不适用）。
> 3. 评估用「测试集总损失」（代价矩阵加权），而非单纯错误率；三法结果宜并列对比。

## 3.2 缺失值与 surrogate

rpart 的两阶段缺失处理：
- **建模时**：主分割变量缺失的样本不参与该分割的 improve 计算；同时搜索 surrogate splits（与主分割一致性最高的其他变量）。
- **预测时**：主分割变量缺失 → 按 `agree` 降序用 surrogate；全缺失 → 按 `usesurrogate` 决定。

### 3.2.1 建模时缺失：improve 的手算对照

将训练集第一行 ShellWeight 置为缺失，用 `maxdepth = 1` 只看根节点分割。

In [ ]:
tr_m <- tr_ab                                       # 复制训练集
tr_m[1, "ShellWeight"] <- NA                        # 制造一个缺失值
with(tr_m, table(Rings, exclude = NULL))             # 确认缺失值被 exclude=NULL 计入统计
with(tr_m, table(cut(ShellWeight, c(0, 0.2545, Inf)), Rings, exclude = NULL)) # 查看缺失值在哪一边

In [ ]:
set.seed(2)                                          # 固定交叉验证随机种子
ct_m <- rpart(Rings ~ ., data = tr_m, method = "class", maxdepth = 1) # 限制深度为 1，只看根节点
summary(ct_m)                                        # 查看根节点的分割详情和 improve

**手算（排除 1 例缺失，共 2087 例）：**

```
improve = 2087·G(1044,1043) − 1166·G(851,315) − 921·G(192,729)
```

注意：原稿曾出现 279.7491（summary 输出）与 279.7941（手算）不一致，根源是**是否把缺失样本计入父节点基数**。rpart 的做法是：父节点 improve 用**排除缺失后**的样本数（2087），左右子节点数也相应为 1166/921（而非 1166/922）。按此口径手算与 summary 一致。

In [ ]:
gini <- function(n1, n2) 1 - (n1/(n1+n2))^2 - (n2/(n1+n2))^2 # 定义 Gini 指数计算函数
imp <- 2087*gini(1044,1043) - 1166*gini(851,315) - 921*gini(192,729) # 按排除缺失后的基数计算
cat("手算 improve =", round(imp, 4), "\n")            # 输出结果与 summary 对照

### 3.2.2 预测时缺失：surrogate 按序使用

用 `maxsurrogate = 7` 让 rpart 搜索全部代理分割，`maxdepth = 1` 便于观察。

In [ ]:
set.seed(2)                                          # 固定交叉验证随机种子
ct_s <- rpart(Rings ~ ., data = tr_ab,
              control = list(maxsurrogate = 7), method = "class", maxdepth = 1) # 强制搜索 7 个代理变量
summary(ct_s)                                        # 查看 Surrogate splits 的 agree 排序

`Surrogate splits` 的 `agree` = 代理分割与原分割一致比例；`adj` = 调整后一致比例。rpart 按 `agree` 降序选用。

**代理搜索算法（重写版）**：对每个候选变量，在排序后的候选切点中找使一致率最高的切点。下面演示数值型变量的搜索：

In [ ]:
# 主分割：ShellWeight < 0.2545 向左
main_left <- tr_ab$ShellWeight < 0.2545              # 记录主分割的左右方向（逻辑向量）

# 对每个数值自变量，搜索使一致率最高的切点
num_vars <- names(tr_ab)[sapply(tr_ab, is.numeric)]  # 提取所有数值型自变量的名字
res <- lapply(num_vars, function(v) {                # 遍历每个数值变量
  x <- tr_ab[[v]]                                    # 提取该变量的取值
  ux <- sort(unique(x))                              # 获取所有唯一值并排序
  cuts <- (ux[-1] + ux[-length(ux)]) / 2             # 计算所有相邻唯一值的中点作为候选切点
  agr <- sapply(cuts, function(c) mean((x < c) == main_left)) # 计算每个切点与原分割的一致率
  data.frame(variable = v, cut = cuts[which.max(agr)], agree = max(agr)) # 返回一致率最高的切点和一致率
})
res <- do.call(rbind, res)                           # 合并所有变量的搜索结果
res[order(-res$agree), ]                             # 按一致率降序排列，与 summary 输出对照

与 `summary(ct_s)` 的 Surrogate splits 顺序一致（WholeWeight → Diameter → LongestShell → …）。

**观察 surrogate 的逐级启用**（以测试集一行为例，依次让主变量缺失）：

In [ ]:
te_s <- te_ab                                        # 复制测试集
te_s[1, "ShellWeight"] <- NA                        # 让主分割变量缺失
predict(ct_s, te_s[1, ], type = "class")             # 此时模型会自动使用 WholeWeight 代理变量进行预测

In [ ]:
te_s[1, "WholeWeight"] <- NA                        # 让第一代理变量也缺失
predict(ct_s, te_s[1, ], type = "class")             # 此时模型自动退而求其次，使用 Diameter 代理

In [ ]:
te_s[1, "Diameter"] <- NA                           # 让第二代理变量也缺失
predict(ct_s, te_s[1, ], type = "class")             # 此时模型使用第三代理变量 LongestShell

**按序缺失时测试误差的变化**（依次将 ShellWeight、WholeWeight、… 整列置 NA）：

In [ ]:
te_s <- te_ab                                        # 重新复制测试集
vars <- c("ShellWeight", "WholeWeight", "Diameter", "LongestShell",
          "VisceraWeight", "ShuckedWeight", "Height", "Type") # 按代理优先级排列变量名
errs <- numeric(length(vars))                        # 初始化误差存储向量
for (k in seq_along(vars)) {                         # 依次让变量缺失
  te_s[[vars[k]]] <- NA                              # 将第 k 个变量整列置为 NA
  errs[k] <- mean(predict(ct_s, te_s, type = "class") != te_s$Rings) # 计算当前的预测错误率
}
data.frame(cum_missing = vars, test_error = round(errs, 4)) # 展示累积缺失下的误差变化

最后一行（全部自变量缺失）的误差 = 多数类比例（L 类占比），验证 `usesurrogate = 2` 的「全缺失按多数类」行为。

> **本节要点**
> 1. 建模时缺失样本不参与 improve 计算；手算需用排除缺失后的基数。
> 2. 预测时按 `agree` 降序用 surrogate；`maxsurrogate` 控制搜索数量，`usesurrogate` 控制全缺失行为。
> 3. 全缺失时 `usesurrogate = 2` 按多数类返回，误差 = 多数类比例。

## 3.3 变量重要性：rpart 自带 vs permutation

### 3.3.1 rpart 自带 importance

定义：变量作为**主变量**的每次分割的 improve 之和，加上作为**代理变量**的 improve × adjusted agreement。输出缩放到总和 100，省略 <1% 的变量。

In [ ]:
barplot(ct_ab$variable.importance, main = "rpart 自带 importance（abalone 分类树）") # 绘制自带变量重要性条形图

### 3.3.2 自带的局限：friedman1 例子

friedman1 数据：$Y = 10\sin(\pi X_1 X_2) + 20(X_3-0.5)^2 + 10X_4 + 5X_5 + \epsilon$，只有 $X_1$–$X_5$ 与 Y 相关，$X_6, X_7$ 无关。

In [ ]:
set.seed(943)                                        # 固定生成数据随机种子
fr <- treemisc::gen_friedman1(1000, nx = 7, sigma = 0.1) # 生成 friedman1 数据，7 个自变量，其中 2 个无关
set.seed(100)                                        # 固定切分随机种子
I <- sample(nrow(fr), nrow(fr) * 0.5)                # 随机抽取一半作为训练集索引
tr_fr <- fr[I, ]; te_fr <- fr[-I, ]                  # 划分训练集和测试集

set.seed(2)                                          # 固定交叉验证随机种子
rt_fr <- rpart(y ~ ., data = tr_fr, cp = 0)          # 建立回归树
rt_fro <- prune_se(rt_fr, prune = TRUE, se = 1)      # 1-SE 剪枝
barplot(rt_fro$variable.importance, main = "rpart 自带 importance（friedman1）") # 画出自带变量重要性

可见自带 importance 未能正确反映 X6、X7 与 Y 无关（它们仍有一定数值）。原因是自带 importance 基于「训练集上的分割 improve」，对过拟合与代理使用敏感。

### 3.3.3 permutation 重要性

原理：对每个变量随机置换其取值，观察预测误差的上升量；上升越多越重要。与因变量无关的变量置换后误差不应显著上升。

In [ ]:
p_load(vip, ggplot2)                                 # 加载计算 permutation 重要性的包
pfun <- function(object, newdata) predict(object, newdata = newdata) # 定义预测包装函数
vis <- vi(rt_fro, method = "permute", train = tr_fr, target = "y",
          metric = "rmse", pred_wrapper = pfun, nsim = 10) # 基于置换法计算重要性
ggplot(vis, aes(x = reorder(Variable, -Importance), y = Importance)) +
  labs(x = "Variable") +                             # 设置 x 轴标签
  geom_bar(stat = "identity") +                      # 绘制条形图
  geom_errorbar(aes(ymin = Importance - StDev, ymax = Importance + StDev), width = 0.5, size = 1) # 绘制误差棒

permutation 法正确地把 X6、X7 排在末位（重要性接近 0）。

> **本节要点**
> 1. rpart 自带 importance 基于训练集 improve，对过拟合/代理敏感，可能高估无关变量。
> 2. permutation importance 基于预测误差，更可靠；两者应结合看。

## 3.4 时间复杂度

三个实验（分类-全类别、分类-全连续、回归）合并为一个函数，只换数据源。预期：每层排序 O(p·n log n)，深度受 cp 控制，总时间近线性于 n。

In [ ]:
time_experiment <- function(data, formula, N = 10, M = 20) {
  nr <- nrow(data)                                   # 获取数据集的总行数
  rc <- matrix(NA, M, N)                             # 创建存储实验结果的矩阵
  pb <- txtProgressBar(style = 3)                    # 创建进度条
  for (n in 1:N) {                                   # 外层循环：逐渐增加数据规模（10% 到 100%）
    for (m in 1:M) {                                 # 内层循环：每种规模重复实验 M 次
      set.seed(n * 1000 + m)                         # 设置随机种子
      d <- data[sample(nr, n * nr / N), ]            # 随机抽样当前规模的数据
      t0 <- Sys.time()                               # 记录开始时间
      rpart(formula, data = d, cp = 0)               # 训练模型
      rc[m, n] <- as.numeric(difftime(Sys.time(), t0, units = "secs")) # 记录耗时
    }
    setTxtProgressBar(pb, n / N)                     # 更新进度条
  }
  boxplot(rc, xlab = "每单位=10% 样本", ylab = "time (s)") # 画出耗时箱线图
  lines(colMeans(rc), type = "o", lwd = 2); grid()   # 画出平均耗时趋势线
  invisible(rc)                                      # 返回结果但不打印
}

In [ ]:
# 分类-全类别
s(mushroom)                                          # 勘察 mushroom 数据
time_experiment(mushroom, Edibility ~ .)             # 运行时间复杂度实验

In [ ]:
# 分类-全连续
set.seed(943)                                        # 固定生成数据随机种子
fr2 <- treemisc::gen_friedman1(1000, nx = 20, sigma = 0.1) # 生成全连续自变量数据
fr2$y <- factor(ifelse(fr2$y < 14.2, 0, 1))          # 将因变量转化为因子以进行分类
time_experiment(fr2, y ~ .)                          # 运行时间复杂度实验

In [ ]:
# 回归
set.seed(943)                                        # 固定生成数据随机种子
fr3 <- treemisc::gen_friedman1(1000, nx = 20, sigma = 0.1) # 生成回归数据
time_experiment(fr3, y ~ .)                          # 运行时间复杂度实验

三条曲线均近线性——rpart 每层排序 O(p·n log n)，树深受 cp 限制，故总时间对 n 近线性。

> **本节要点**
> 1. rpart 训练时间对样本量近线性；自变量数 p 通过每层排序影响常数。
> 2. 类别自变量需枚举二分组合，p 大时比连续自变量慢。

# 附录

## A. predict 输出速查

| method | `type` | 返回 |
|---|---|---|
| `class` | `"class"`（默认） | 类别因子 |
| `class` | `"prob"` | 概率矩阵（列=levels） |
| `anova` | `"vector"`（默认） | 预测均值向量 |
| `anova` | `"matrix"` | nsplit × n 矩阵 |
| `poisson` | `"vector"`（默认） | 预测计数均值 |
| `poisson` | `"matrix"` | nsplit × n 矩阵 |
| `exp`（生存） | `"vector"`（默认） | 预测生存时间/风险 |
| `exp` | `"matrix"` | nsplit × n 矩阵 |

## B. 常见坑

1. **xerror/xstd 有随机性**：复现请在 `rpart()` 前 `set.seed()`。
2. **计数因变量不会自动走 poisson**：会被当作连续而走 `anova`，必须显式 `method = "poisson"`；二分类若因变量是 0/1 数值也会走 `anova`，需先转因子。
3. **`cp = 0` 在代价敏感场景可能不稳定**：用 `cp = 1e-5`；第 1 节普通场景的 cp=0 不适用于代价敏感。
4. **`plotcp` 横轴是相邻 cp 的几何平均**，不是 cp 本身。
5. **`prune_se()` 来自 treemisc 包**，`se = 1` 即标准 1-SE 规则。
6. **缺失值手算 improve**：父节点基数用排除缺失后的样本数。
7. **自带 variable.importance 可能高估无关变量**：结合 permutation importance。
8. **`usesurrogate` 默认 2**：全缺失按多数类返回，误差 = 多数类比例。
9. **分类树默认 Gini**：可用 `parms = list(split = "information")` 改用信息增益；`parms` 是 `rpart()` 的直接参数，不属于 `rpart.control`。
10. **`post()` 已过时**：生成 PostScript 图，现代环境建议用 `rpart.plot`/`partykit` 替代。

<a id="appendix-c"></a>
## C. 关于 1-SE 规则的深入探讨

**1-SE 规则** 是决策树剪枝时最核心的策略：令 `thr = min(xerror) + 1 * xstd[对应行]`，在满足 `xerror ≤ thr` 的所有 cp 中取**最大**的 cp。

为什么要这样设定呢？这背后蕴含着严谨的统计学逻辑和对抗方差的智慧：

### 1. 为什么不用“绝对最低误差”？（警惕运气成分）
在 10 折交叉验证中，数据是随机切分的。假设模型 A 的测试误差是 0.40，模型 B 是 0.41。你能笃定 A 一定比 B 好吗？不一定。这 0.01 的差距很可能仅仅是因为 B 刚好分到了几个比较难预测的“奇葩样本”。如果换一种切法，结果可能逆转。这就是**随机波动（方差）**。

### 2. `xstd` 是什么？它是“容错空间”
`xstd`（交叉验证标准差）恰恰量化了这种随机波动。如果 `xstd` 很大，说明 10 次验证误差忽高忽低，模型不稳定。因此，统计学家引入 `thr = min(xerror) + 1 * xstd` 的潜台词是：**“我不要求你达到最完美的成绩（最低误差），只要你的成绩在完美成绩的 1 个标准误波动范围内，我就认为你是合格的。”**

### 3. 为什么正好是“1”倍？（1-SE 规则的科学依据）
在统计学中，**“1 个标准差（1-SE）”** 是一个非常经典的置信区间指标。
*   **0-SE**（不加标准差）：苛求绝对最低误差，会选出极其复杂的树（过拟合）。
*   **2-SE**：容错空间太大，可能选出一棵过于简单的树（欠拟合）。
*   **1-SE**：是**奥卡姆剃刀（Occam's Razor）**的最佳平衡点——**在模型性能损失极小（不显著）的前提下，选择最简单的模型。**

### 4. 生活化的比喻
假设你要招募一个长跑运动员：
*   **A 选手**：平均成绩 10.0 秒，但忽好忽坏（标准差 0.5 秒）。
*   **B 选手**：平均成绩 10.2 秒，但极其稳定（标准差 0.1 秒）。

如果只选“最低均值”，你会选 A。但你可能会想：*“A 的 10.0 秒可能只是运气好。为了保险，我要设置一个门槛：只要成绩在 10.0 + 0.5 = 10.5 秒以内的，我都考虑。”* 这个时候，B 选手（10.2 秒）就进入了门槛。如果此时你还发现 B 选手**要的薪水更少（对应模型更简单、节点更少、更好解释）**，你就会果断放弃 A，选择 B。

**在决策树中：**
*   “薪水更少、更好管理” = **CP 值更大，树更简单**。
*   “成绩达标” = **xerror 在最低值 + 1个标准差的范围内**。

所以，设定 `thr = xerror_min + xstd` 是为了**用微小的性能让步，换取模型极大的简化，同时规避了评估指标随机波动带来的假象**。

<a id="appendix-d"></a>
## D. rpart 其余函数与数据集速览（补全包内全部对象）

正文只讲了主干函数。本节把 `rpart` 包内**未被正文展开**的函数与数据集按用途归类速览，保证对包内全部对象有交代；每一项只给一句话用途，需要时再查 `?函数名`。

### D.1 诊断类（看模型哪里不对）

| 函数 | 用途 | 典型用法 |
|---|---|---|
| `residuals.rpart()` | 提取每个样本的残差（分类为误判指示，回归为观测减预测，泊松/生存为偏差残差） | `resid(fit)` |
| `meanvar.rpart()` / `meanvar()` | 画各节点的均值-方差图，观察不同节点的响应分布是否同质 | `meanvar(fit)` |
| `rsq.rpart()` | 画不同分裂数下的近似 R²，直观看到「再加分裂还能提升多少」 | `rsq.rpart(fit)` |

### D.2 交叉验证类（与 cp 表同源）

| 函数 | 用途 | 典型用法 |
|---|---|---|
| `xpred.rpart()` | 返回交叉验证预测值（cp 表里的 xerror 就是由它汇总而来），可用于自定义 CV 诊断 | `xpred.rpart(fit, cp = 0.01)` |

### D.3 绘图底层类（plot/text 的零件）

| 函数 | 用途 | 典型用法 |
|---|---|---|
| `labels.rpart()` | 生成每个节点的分割标签文本，`text.rpart` 内部即调用它 | `labels(fit)` |
| `post.rpart()` / `post()` | 输出 PostScript 格式树图（老式打印用，现代环境建议用 `rpart.plot`/`partykit` 替代） | `post(fit, file = "tree.ps")` |

### D.4 内部/辅助类（了解即可）

| 函数/对象 | 用途 |
|---|---|
| `na.rpart()` | 建树前处理缺失值的内部函数：标记哪些行因缺失被排除，是 3.2 节缺失机制的底层实现 |
| `rpart.exp()` | 指数（生存）拟合的初始化函数，2.4 节 `method = "exp"` 的底层入口 |
| `rpart.object` | 树对象的结构定义（`$cptable`、`$variable.importance`、`$splits` 等字段的文档），读对象字段时查它 |

### D.5 数据集补全

正文用过 `solder.balance`（2.3）、`stagec`（2.4），0.2 已列 `kyphosis`、`solder`、`car.test.frame`。包内**全部数据集**如下：

| 数据集 | 内容 | 教程中的使用 |
|---|---|---|
| `kyphosis` | 儿童脊柱矫正手术后是否出现后凸畸形的数据（二分类经典例） | 仅列名 |
| `solder` / `solder.balance` | 印刷电路板焊接缺陷数据；`solder.balance` 为其平衡版本 | 2.3 泊松树 |
| `stagec` | 前列腺癌 Stage C 生存数据（含随访时间与状态） | 2.4 生存树 |
| `car.test.frame` | 《Consumer Reports》1990 汽车数据（测试版） | 仅列名 |
| `car90` | 同源汽车数据（更完整版本，含更多变量与缺失） | 未用 |
| `cu.summary` | 同源汽车数据的摘要子集 | 未用 |

> 说明：`car.test.frame`、`car90`、`cu.summary` 是同一组汽车数据的三个版本，选一个用于练习即可；`kyphosis` 是二分类树的最小示例，适合快速试手。

### D.6 小结

- **主干已讲透**：`rpart`、`rpart.control`、`predict`、`printcp`/`plotcp`、`prune`、`summary`、`print`、`plot`/`text`、`path`/`snip`。
- **本节补全**：`residuals`、`meanvar`、`rsq.rpart`、`xpred`、`labels`、`post`、`na.rpart`、`rpart.exp`、`rpart.object`，以及 `car90`、`cu.summary` 数据集。
- 至此，`rpart` 包内**全部函数与数据集**均有交代：常用的在正文深入，少用的在此速览，按需回查 `?函数名` 即可。